In [1]:
import numpy as np
import matplotlib.pyplot as plt
import imodex_miri_postprocess_pipeline as imp #homemade pipeline
from scipy import ndimage
from scipy.optimize import curve_fit
from astropy.stats import sigma_clip
from astropy.io import fits
import glob
from scipy.optimize import minimize
#from collections import defaultdict
from scipy import ndimage
import math
import glob, os
import scipy.optimize as optimize
from matplotlib.colors import LogNorm

#ploting
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter

def flux_uJy_to_Wm(wl,flux) :
    return np.array(flux)/(333564095198152064*np.array(wl)**2)

def flux_Wm_to_uJy(wl,flux) :
    return np.array(flux)*(333564095198152064*np.array(wl)**2)

In [2]:
################
#### F1140C ####
################


path_save = '/home/cperrot/Work/projet/JWST/Target/GO/P01668/HD92945/'

if not os.path.exists(path_save):
    os.makedirs(path_save)

pixscale = 0.110 # arcsec
filt='F1140C'

if filt == 'F1065C':
    lam = 10.65e-6    # lambda en m
    x0  = 120.25
    y0  = 112.03

if filt == 'F1140C':
    lam = 11.40e-6    # lambda en m
    x0  = 119.77
    y0  = 112.16

if filt == 'F1550C':
    lam = 15.50e-6    # lambda en m
    x0  = 119.71
    y0  = 113.28


aff  = 'cal'

path_tar_1  = '/home/cperrot/Work/raw/JWST/Target/MAST/CAL/P01668/obs007/'
path_tar_2  = '/home/cperrot/Work/raw/JWST/Target/MAST/CAL/P01668/obs008/'
path_ref    = '/home/cperrot/Work/raw/JWST/Target/MAST/CAL/P01668/obs009/'

path_bkg_1  = '/home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/P01668/obs006/'
path_bkg_2  = '/home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/P01668/obs010/'


cube_tar_1 ,header_tar_1   = imp.load_cube(path_tar_1,aff)
cube_tar_2 ,header_tar_2   = imp.load_cube(path_tar_2,aff)
cube_ref   ,header_ref     = imp.load_cube(path_ref,aff)
 
min_bkg_1 = fits.open(glob.glob(path_bkg_1+'P*_master_bkg_cal.fits',recursive=False)[0])[0].data
min_bkg_2 = fits.open(glob.glob(path_bkg_2+'P*_master_bkg_cal.fits',recursive=False)[0])[0].data
    
#bkg subtraction
cube_tar_1 = cube_tar_1-min_bkg_1
cube_tar_2 = cube_tar_2-min_bkg_1
cube_ref   = cube_ref-min_bkg_2

if False:
    ### write SCI and REF fits after minimum bkg subtraction  
    hdu_list = fits.HDUList([fits.PrimaryHDU(header=header_tar_1[0]),fits.ImageHDU(data=cube_tar_1,header=header_tar_1[0])])
    hdu_list.writeto(path_save+'HD92945_roll-1_dither1_minimum_bkg_cal.fits',overwrite=True)
    hdu_list = fits.HDUList([fits.PrimaryHDU(header=header_tar_2[0]),fits.ImageHDU(data=cube_tar_2,header=header_tar_2[0])])
    hdu_list.writeto(path_save+'HD92945_roll-2_dither1_minimum_bkg_cal.fits',overwrite=True)
    hdu_list = fits.HDUList([fits.PrimaryHDU(header=header_ref[0][0]),fits.ImageHDU(data=cube_ref,header=header_ref[0][1])])
    hdu_list.writeto(path_save+'REF_HD95234_dither1-9_minimum_bkg_cal.fits',overwrite=True)  


clipval = 3.
cube_tar_1  = imp.sigma_clip_jwst(cube_tar_1,clipval)
cube_tar_2  = imp.sigma_clip_jwst(cube_tar_2,clipval)
cube_ref    = imp.sigma_clip_jwst(cube_ref,clipval)

cube_tar_1 = imp.clean_image_nan(cube_tar_1)
cube_tar_2 = imp.clean_image_nan(cube_tar_2)
cube_ref = imp.clean_image_nan(cube_ref)

#resize
newd = 200
xd = np.shape(cube_tar_1)[1]
yd = np.shape(cube_tar_1)[0]
scale=[(newd/2.)*pixscale,-1*(newd/2.)*pixscale,-1*(newd/2.)*pixscale,(newd/2.)*pixscale]

cube_tar_1 = ndimage.shift(cube_tar_1,[newd//2-y0,newd//2-x0])
cube_tar_2 = ndimage.shift(cube_tar_2,[newd//2-y0,newd//2-x0])
for i in range(len(cube_ref)): cube_ref[i,:,:] = ndimage.shift(cube_ref[i,:,:],[newd//2-y0,newd//2-x0])
y0 = newd//2
x0 = newd//2

cube_tar_1 = cube_tar_1[y0-newd//2:y0+newd//2,x0-newd//2:x0+newd//2]
cube_tar_2 = cube_tar_2[y0-newd//2:y0+newd//2,x0-newd//2:x0+newd//2]
cube_ref   = cube_ref[:,y0-newd//2:y0+newd//2,x0-newd//2:x0+newd//2]

    

#PCA # commented because PCA is bad for this target
#PCA_1140_1 = imp.sub_PCA(cube_tar_1, cube_ref, [newd//2,newd//2], 2, dim=newd//2, plot=False)
#PCA_1140_2 = imp.sub_PCA(cube_tar_2, cube_ref, [newd//2,newd//2], 2, dim=newd//2, plot=False)

# flux scaling for RDI
f_scale_1 = imp.flux_scaling_ta(path_tar_1+'TA', path_ref+'TA', aff, lam)
f_scale_2 = imp.flux_scaling_ta(path_tar_2+'TA', path_ref+'TA', aff, lam)
print("f_scale_1 with TA : ",f_scale_1)
print("f_scale_2 with TA : ",f_scale_2)

#perform RDI 
RDI_1140_1 = imp.RDI_classic(cube_tar_1, cube_ref, f_scale=f_scale_1)
RDI_1140_2 = imp.RDI_classic(cube_tar_2, cube_ref, f_scale=f_scale_2)

#subshift for roll 1
#RDI_1140_1 = RDI_classic(ndimage.shift(clean_image_nan(cube_tar_1),[0.0,0.1]), clean_image_nan(cube_ref), f_scale=f_scale_1)

#rotation
PA_V3_1 = header_tar_1[1]['PA_V3']     # 59.58852013851359
V3i_1   = header_tar_1[1]['V3I_YANG']  # 4.83544897
PA_V3_2 = header_tar_2[1]['PA_V3']     # 66.59658743620635
V3i_2   = header_tar_2[1]['V3I_YANG']  # 4.83544897

#PCA_1140_1 = ndimage.rotate(PCA_1140_1, -(PA_V3_1+V3i_1), reshape=False)
#PCA_1140_2 = ndimage.rotate(PCA_1140_2, -(PA_V3_2+V3i_2), reshape=False)
RDI_1140_1_c = np.copy(RDI_1140_1)
RDI_1140_2_c = np.copy(RDI_1140_2)
RDI_1140_1_c = imp.clean_image_nan(RDI_1140_1_c)
RDI_1140_2_c = imp.clean_image_nan(RDI_1140_2_c)
RDI_1140_1_r = ndimage.rotate(RDI_1140_1_c, -(PA_V3_1+V3i_1), reshape=False)
RDI_1140_2_r = ndimage.rotate(RDI_1140_2_c, -(PA_V3_2+V3i_2), reshape=False)

##PCA_1140_t = (PCA_1140_1+PCA_1140_2)/2.
RDI_1140_t   = (RDI_1140_1+RDI_1140_2)/2.
RDI_1140_t_c = (RDI_1140_1_c+RDI_1140_2_c)/2.
RDI_1140_t_r = (RDI_1140_1_r+RDI_1140_2_r)/2.


imp.save_fits(RDI_1140_1,path_save+'RDI_1140_roll_1.fits')
imp.save_fits(RDI_1140_2,path_save+'RDI_1140_roll_2.fits')
imp.save_fits(RDI_1140_t,path_save+'RDI_1140_somme_roll.fits')
imp.save_fits(RDI_1140_1_r,path_save+'RDI_1140_roll_1_rot.fits')
imp.save_fits(RDI_1140_2_r,path_save+'RDI_1140_roll_2_rot.fits')
imp.save_fits(RDI_1140_t_r,path_save+'RDI_1140_somme_roll_rot.fits')

imp.save_fits(cube_tar_1,path_save+'cube_1140_pixelcorr_1.fits')
imp.save_fits(cube_tar_2,path_save+'cube_1140_pixelcorr_2.fits')

imp.save_fits(RDI_1140_1_c,path_save+'RDI_1140_roll_1_clean.fits')
imp.save_fits(RDI_1140_2_c,path_save+'RDI_1140_roll_2_clean.fits')
imp.save_fits(RDI_1140_t_c,path_save+'RDI_1140_somme_roll_clean.fits')
imp.save_fits(imp.clean_image_nan(cube_tar_1),path_save+'cube_1140_pixelcorr_1_clean.fits')
imp.save_fits(imp.clean_image_nan(cube_tar_2),path_save+'cube_1140_pixelcorr_2_clean.fits')




Loading of 1 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 288 / 224
Loading of 1 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 288 / 224
Loading of 9 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 288 / 224


/home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1218: RuntimeWarning: All-NaN slice encountered
  r, k = function_base._ureduce(a, func=_nanmedian, axis=axis, out=out,
/home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Loading of 2 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 200 / 200
sigma model PSF:  1.3965812874037857
157 156
Roll - TA 1
Result : A = 21748.5 / x0 = 155.94 / y0 = 156.78 / width_x = 1.37 / width_y = 1.43
Stddev : A = 53.9 / x0 = 0.0 / y0 = 0.0 / width_x = 0.0 / width_y = 0.0
115 117
Roll - TA 2
Result : A = 21658.1 / x0 = 116.61 / y0 = 114.58 / width_x = 1.36 / width_y = 1.43
Stddev : A = 53.1 / x0 = 0.0 / y0 = 0.0 / width_x = 0.0 / width_y = 0.0
Loading of 2 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 200 / 200
157 156
Ref - TA 1
Result : A = 984002.6 / x0 = 155.95 / y0 = 157.11 / width_x = 1.41 / width_y = 1.49
Stddev : A = 2577.7 / x0 = 0.0 / y0 = 0.0 / width_x = 0.0 / width_y = 0.0
115 117
Ref - TA 2
Result : A = 984228.7 / x0 = 116.62 / y0 = 114.58 / width_x = 1.4 / width_y = 1.48
Stddev : A = 2526.4 / x0 = 0.0 / y0 = 0.0 / width_x = 0.0 / width_y = 0.0
Loading of 2 cubes.
Numb